In [ ]:
# ====================== 导入库 & 全局配置 ======================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# 绘图中文设置
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 数据根目录 = 当前脚本所在目录
DATA_ROOT = Path(__file__).parent

# ====================== 读取RFM基础表 ======================
rfm = pd.read_csv(DATA_ROOT / "rfm_base.csv", parse_dates=["last_purchase_time"])

# ====================== 1. RM二维分档规则 ======================
# R维度：活跃度划分，以180天（半年）为界，半年内有购买视为活跃用户
R_THRESHOLD_DAYS = 180
rfm["R_level"] = np.where(rfm["recency_days"] <= R_THRESHOLD_DAYS, "活跃", "沉睡")

# M维度：价值划分，以消费金额中位数为界，高于中位数为高价值用户
M_THRESHOLD = rfm["monetary"].median()
rfm["M_level"] = np.where(rfm["monetary"] >= M_THRESHOLD, "高价值", "低价值")

print(f"===== RM分层阈值设定 =====")
print(f"活跃度阈值：{R_THRESHOLD_DAYS} 天（≤阈值=活跃，>阈值=沉睡）")
print(f"价值阈值：{M_THRESHOLD:.2f} 雷亚尔（≥阈值=高价值，<阈值=低价值）")


# ====================== 2. 二维交叉用户分层 ======================
def rm_segment(row):
    r_level = row["R_level"]
    m_level = row["M_level"]

    if r_level == "活跃" and m_level == "高价值":
        return "高价值活跃用户"
    elif r_level == "活跃" and m_level == "低价值":
        return "低价值活跃用户"
    elif r_level == "沉睡" and m_level == "高价值":
        return "高价值沉睡用户"
    else:
        return "低价值沉睡用户"


rfm["用户分层"] = rfm.apply(rm_segment, axis=1)

# ====================== 3. 分层结果统计 ======================
# 按业务优先级排序
seg_order = ["高价值活跃用户", "低价值活跃用户", "高价值沉睡用户", "低价值沉睡用户"]
seg_count = rfm["用户分层"].value_counts().reindex(seg_order)
seg_pct = (seg_count / len(rfm) * 100).round(2)

print("\n===== RM用户分层结果 =====")
for seg in seg_order:
    print(f"{seg}：{seg_count[seg]:,} 人，占比 {seg_pct[seg]}%")

# 各分层核心指标明细
print("\n===== 各层核心指标 =====")
seg_detail = rfm.groupby("用户分层").agg(
    用户数=("customer_id", "count"),
    平均最近消费天数=("recency_days", "mean"),
    平均消费总金额=("monetary", "mean"),
    平均购买频次=("frequency", "mean")
).round(2)
print(seg_detail.reindex(seg_order).to_string())

# 数据特征校验
repeat_cnt = (rfm["frequency"] > 1).sum()
print(f"\n===== 数据集特征校验 =====")
print(f"总用户数：{len(rfm):,} 人")
print(f"复购用户（≥2次）：{repeat_cnt:,} 人，占比 {repeat_cnt / len(rfm) * 100:.2f}%")

# ====================== 4. 可视化（兼容高版本Seaborn，无警告） ======================
plt.figure(figsize=(10, 6))
# 四层对应配色：核心-新客-召回-低价值
bar_colors = ["#2c3e50", "#27ae60", "#f39c12", "#95a5a6"]

# hue+x一致+legend=False，彻底消除palette弃用警告
ax = sns.barplot(
    x=seg_count.index,
    y=seg_count.values,
    hue=seg_count.index,
    palette=bar_colors,
    legend=False
)

# 柱子上标注数值与占比
max_val = seg_count.max()
for i, (cnt, pct) in enumerate(zip(seg_count.values, seg_pct.values)):
    ax.text(i, cnt + max_val * 0.02, f"{cnt:,}\n({pct}%)", ha='center', fontsize=10)

plt.title("RM 二维用户价值分层分布", fontsize=14)
plt.ylabel("用户数量", fontsize=12)
plt.xlabel("用户分层", fontsize=12)
plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.savefig(DATA_ROOT / "RM用户分层分布图.png", dpi=300, bbox_inches='tight')
plt.show()

# ====================== 5. 导出结果 ======================
rfm.to_csv(DATA_ROOT / "rm_segmented.csv", index=False, encoding="utf-8-sig")
print("\n✅ 分层结果已导出：rm_segmented.csv")
print("✅ 分层统计图已导出：RM用户分层分布图.png")